In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path.cwd().parents[0] / "src"))

from utils import (
    COL_CATEGORY,
    COL_CITY,
    COL_DISTRICT,
    add_time_features,
    apply_basic_qc,
    load_seoul_data,
    translate_categories,
)

project_root = Path.cwd().parents[0]
data_path = project_root / "data" / "fire_rescue_seoul_2021.csv"
fig_dir = project_root / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_path


In [ ]:
df = load_seoul_data(data_path)

print("Shape:", df.shape)
df.head()

In [ ]:
expected_cols = [
    "번호",
    "신고년월일", "신고시각",
    "출동년월일", "출동시각",
    "발생장소_시", "발생장소_구", "발생장소_동",
    "사고원인", "사고원인코드명_사고종별",
]

missing = [c for c in expected_cols if c not in df.columns]
print("Missing columns:", missing)

df[expected_cols].isna().mean().sort_values(ascending=False).head(10)


In [ ]:
df_feat = add_time_features(df)

df_feat[["report_dt", "dispatch_dt", "dispatch_delay_min", "month", "hour", "dow"]].head()


In [ ]:
clean, excluded = apply_basic_qc(df_feat)

print("Total records:", len(df_feat))
print("Clean records:", len(clean))
print("Excluded records:", len(excluded))
print("Excluded share (%):", round(100 * len(excluded) / len(df_feat), 3))

clean["dispatch_delay_min"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


In [ ]:
monthly = clean.groupby(clean["report_dt"].dt.to_period("M")).size().sort_index()

plt.figure(figsize=(8, 4.5))
plt.plot(monthly.index.astype(str), monthly.values, marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Seoul Rescue Activity Volume by Month (2021)")
plt.xlabel("Month")
plt.ylabel("Number of records")
plt.tight_layout()

out = fig_dir / "monthly_volume.png"
plt.savefig(out, dpi=200)
out


In [ ]:
top_cat = clean[COL_CATEGORY].value_counts().head(12)
top_cat.index = translate_categories(top_cat.index.to_series())
top_cat = top_cat.sort_values()

plt.figure(figsize=(8, 5.5))
plt.barh(top_cat.index, top_cat.values)
plt.title("Top Incident Categories in Seoul (2021)")
plt.xlabel("Count")
plt.tight_layout()

out = fig_dir / "top_categories_eng.png"
plt.savefig(out, dpi=200)
out


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(clean["dispatch_delay_min"].clip(upper=20), bins=40)
plt.title("Dispatch Delay Distribution (clipped at 20 minutes)")
plt.xlabel("Minutes from report to dispatch")
plt.ylabel("Frequency")
plt.tight_layout()

out = fig_dir / "dispatch_delay_hist.png"
plt.savefig(out, dpi=200)
out


In [ ]:
pivot = clean.pivot_table(
    index=clean["report_dt"].dt.day_name(),
    columns=clean["report_dt"].dt.hour,
    values="번호",
    aggfunc="count",
    fill_value=0,
)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex(order)

plt.figure(figsize=(11, 4.5))
plt.imshow(pivot.values, aspect="auto")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(0, 24, 2), list(range(0, 24, 2)))
plt.title("Incident Volume Heatmap (Day of Week x Hour)")
plt.xlabel("Hour of day")
plt.ylabel("Day of week")
plt.colorbar(label="Count")
plt.tight_layout()

out = fig_dir / "dow_hour_heatmap.png"
plt.savefig(out, dpi=200)
out


In [ ]:
top_for_box = clean[COL_CATEGORY].value_counts().head(8).index
box_data = [clean.loc[clean[COL_CATEGORY] == c, "dispatch_delay_min"] for c in top_for_box]
labels = list(translate_categories(top_for_box.to_series()))

plt.figure(figsize=(10, 5))
plt.boxplot(box_data, labels=labels, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.title("Dispatch Delay by Category (Top 8, outliers hidden)")
plt.ylabel("Minutes")
plt.tight_layout()

out = fig_dir / "delay_by_category_box_eng.png"
plt.savefig(out, dpi=200)
out
